# Block Data Modeling

Notebook dedicado apenas ao treino e avalia??o dos modelos sobre um dataset j? preparado.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import seaborn as sns

from src.data.blockchain_loader import load_asset_dataframe
from src.data.blockchain_preprocessing import BlockchainPreprocessingConfig, prepare_blockchain_dataframe
from src.features.blockchain_targets import DEFAULT_HORIZONS, add_direction_targets
from src.models.blockchain_training import (
    DEFAULT_IMPORTANCE_THRESHOLDS,
    build_default_models,
    evaluate_models_by_horizon,
    evaluate_models_with_feature_selection,
)


In [ ]:
DATA_DIR = PROJECT_ROOT / 'data' / 'raw' / 'Blockchain_data'
ASSET_NAME = 'btc'
HORIZONS = DEFAULT_HORIZONS
MODELS = build_default_models()


In [ ]:
raw_btc = load_asset_dataframe(DATA_DIR, ASSET_NAME)
config = BlockchainPreprocessingConfig(start_date='2017-08-17')
btc_base = prepare_blockchain_dataframe(raw_btc, config=config)
btc_prepared = add_direction_targets(btc_base, price_column='PriceUSD', horizons=HORIZONS)

btc_prepared.shape


In [ ]:
baseline_results, baseline_artifacts = evaluate_models_by_horizon(
    df=btc_prepared,
    horizons=HORIZONS,
    models=MODELS,
)

baseline_results


In [ ]:
selected_results, selected_artifacts = evaluate_models_with_feature_selection(
    df=btc_prepared,
    horizons=HORIZONS,
    models=MODELS,
    importance_thresholds=DEFAULT_IMPORTANCE_THRESHOLDS,
)

selected_results


In [ ]:
for model_name, horizon_artifacts in selected_artifacts.items():
    for horizon, artifact in horizon_artifacts.items():
        print(f'{model_name} | {horizon} | {len(artifact['selected_features'])} features')
        display(artifact['importance'].head(10))


In [ ]:
model_name = 'XGBoost'
horizon = 't+1'
confusion = selected_artifacts[model_name][horizon]['last_fold'].confusion

plt.figure(figsize=(6, 4))
sns.heatmap(confusion, annot=True, fmt='d', cmap='Blues')
plt.title(f'Confusion Matrix - {model_name} {horizon}')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()
